# IAN 620 · Complete LangChain Tutorial
## From the fundamentals to RAG, tools, agents, Chroma, DuckDuckGo, and LangGraph

This notebook is a standalone LangChain tutorial. It starts with the smallest building blocks and gradually assembles them into useful AI applications.

### What you will learn

1. What LangChain is—and what it is not.
2. LangChain vs. LangGraph.
3. Models, messages, prompts, and output parsers.
4. LangChain Expression Language (LCEL) and Runnables.
5. `invoke`, `batch`, and `stream`.
6. `Document` objects, loaders, metadata, and text splitting.
7. Local Hugging Face embeddings.
8. Chroma vector storage and semantic search.
9. Retrievers and classic RAG.
10. Tools with `@tool`.
11. DuckDuckGo web search.
12. Tool calling and the manual agent loop.
13. `create_agent`.
14. Structured output.
15. Conversation history.
16. Optional local Ollama integration.
17. How LangChain fits with LangGraph.

> **No paid API key is required for the core notebook.** The default examples download an open Hugging Face model and local embedding model. Provider-based examples are clearly marked optional.

## 0. What is LangChain?

LangChain is a framework for composing language-model applications from reusable components.

At a practical level, it gives us common interfaces for:

- chat models and LLMs,
- messages and prompts,
- output parsers,
- documents and document loaders,
- text splitters,
- embeddings,
- vector stores and retrievers,
- tools,
- agents,
- middleware and integrations.

LangChain is **not** a model and it is **not** a vector database. It connects and organizes components that may come from many different libraries or providers.

### Current mental model

**Model + prompt + tools + harness = agent application**

The current LangChain `create_agent` API is a configurable harness around a model/tool loop. LangGraph is the lower-level orchestration layer used when you need finer control over state, branches, loops, persistence, or human checkpoints.

### LangChain vs. LangGraph

| Use case | Start with |
|---|---|
| Prompt a model | LangChain model + prompt |
| Build reusable chains | LangChain Runnables / LCEL |
| Build a normal RAG pipeline | LangChain retriever + prompt + model |
| Build a simple tool-using agent | LangChain `create_agent` |
| Build explicit state machines, loops, retries, or mixed deterministic/agentic workflows | LangGraph |
| Need both integrations and custom orchestration | LangChain components inside LangGraph nodes |

LangChain and LangGraph are complementary rather than competitors.

## 1. Install the tutorial packages

The core notebook uses a local Hugging Face model, local embeddings, Chroma, and DuckDuckGo.

Run this cell once in Colab.

In [ ]:
!pip -q install -U \
    langchain \
    langgraph \
    langchain-community \
    langchain-huggingface \
    langchain-chroma \
    langchain-text-splitters \
    chromadb \
    "sentence-transformers>=5.2.0" \
    "transformers>=4.52.4" \
    accelerate \
    duckduckgo-search \
    pydantic

## 2. Imports and version check

LangChain is split into a small core package plus integration packages. This keeps provider-specific dependencies separate from the core abstractions.

In [ ]:
import os
import shutil
import urllib.request
from pathlib import Path

import pandas as pd
import torch

import langchain
import langgraph
import chromadb

print('LangChain:', langchain.__version__)
print('LangGraph installed:', bool(langgraph))
print('Chroma:', chromadb.__version__)
print('CUDA available:', torch.cuda.is_available())

## 3. LangChain's most important abstraction: a Runnable

Many LangChain components implement the **Runnable** interface.

That gives them a common set of methods such as:

- `.invoke(input)` — run once,
- `.batch(inputs)` — run several inputs,
- `.stream(input)` — stream results when supported,
- `.ainvoke(...)` / `.abatch(...)` — asynchronous versions,
- `|` — compose Runnables into a pipeline.

This shared interface is what makes LangChain components easy to connect.

## 4. Load a local language model with LangChain

We use a downloadable Hugging Face model so the core tutorial does not require an API key.

On a GPU runtime, the notebook uses a slightly larger model. On CPU, it uses the smaller version.

> In Colab: **Runtime → Change runtime type → GPU** is recommended.

In [ ]:
from langchain_huggingface import ChatHuggingFace, HuggingFacePipeline

LOCAL_MODEL = (
    "Qwen/Qwen3-1.7B"
    if torch.cuda.is_available()
    else "Qwen/Qwen3-0.6B"
)

hf_llm = HuggingFacePipeline.from_model_id(
    model_id=LOCAL_MODEL,
    task="text-generation",
    device_map="auto",
    model_kwargs={
        "torch_dtype": "auto",
    },
    pipeline_kwargs={
        "max_new_tokens": 180,
        "do_sample": False,
        "return_full_text": False,
    },
)

chat_model = ChatHuggingFace(llm=hf_llm)

print('Loaded:', LOCAL_MODEL)

### Invoke the model directly

Chat models accept message-like input and return an `AIMessage`.

In [ ]:
response = chat_model.invoke("Explain LangChain in two short sentences.")
print(response.content)

## 5. Messages

Chat applications normally use different message roles.

- `SystemMessage` — instructions or role.
- `HumanMessage` — user input.
- `AIMessage` — model output.
- `ToolMessage` — a tool result returned to the model.

In [ ]:
from langchain.messages import SystemMessage, HumanMessage, AIMessage

messages = [
    SystemMessage(
        content="You are a concise Python instructor."
    ),
    HumanMessage(
        content="What is a Python dictionary?"
    ),
]

reply = chat_model.invoke(messages)
print(reply.content)

## 6. Prompt templates

Prompt templates let us separate reusable instructions from runtime values.

Use `PromptTemplate` for plain text and `ChatPromptTemplate` for message-oriented prompts.

In [ ]:
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate.from_template(
    """Explain {topic} to a beginner.
Give one simple example.
Keep the answer under {words} words."""
)

formatted = prompt.invoke({
    "topic": "vector embeddings",
    "words": 100,
})

print(formatted.text)

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

chat_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are a helpful instructor for {course}.",
    ),
    (
        "human",
        "Explain {topic} with a short example.",
    ),
])

prompt_value = chat_prompt.invoke({
    "course": "natural language processing",
    "topic": "cosine similarity",
})

print(prompt_value)

## 7. LCEL: connect components with `|`

LangChain Expression Language (LCEL) uses the pipe operator to compose Runnables.

Conceptually:

`input → prompt → model → parser`

In [ ]:
from langchain_core.output_parsers import StrOutputParser

simple_chain = (
    chat_prompt
    | chat_model
    | StrOutputParser()
)

answer = simple_chain.invoke({
    "course": "NLP",
    "topic": "TF-IDF",
})

print(answer)

### Why use a parser?

A model returns an `AIMessage`. `StrOutputParser()` converts the model response into a normal Python string.

Parsers are also useful for JSON or typed outputs.

## 8. `invoke`, `batch`, and `stream`

Because chains use the Runnable interface, the same chain can be executed in several ways.

In [ ]:
one = simple_chain.invoke({
    "course": "NLP",
    "topic": "stemming",
})

print(one)

In [ ]:
questions = [
    {"course": "NLP", "topic": "tokenization"},
    {"course": "NLP", "topic": "lemmatization"},
    {"course": "NLP", "topic": "n-grams"},
]

batch_answers = simple_chain.batch(questions)

for answer in batch_answers:
    print('---')
    print(answer)

In [ ]:
print('Streaming example:\n')

for chunk in simple_chain.stream({
    "course": "NLP",
    "topic": "word embeddings",
}):
    print(chunk, end='', flush=True)

## 9. Runnables beyond models

`RunnableLambda`, `RunnablePassthrough`, and `RunnableParallel` are useful when normal Python functions need to participate in a LangChain pipeline.

In [ ]:
from langchain_core.runnables import (
    RunnableLambda,
    RunnablePassthrough,
    RunnableParallel,
)


def clean_text(text):
    return ' '.join(text.strip().split())


cleaner = RunnableLambda(clean_text)

print(cleaner.invoke('  LangChain    is   composable.  '))

In [ ]:
parallel = RunnableParallel({
    "original": RunnablePassthrough(),
    "upper": RunnableLambda(lambda x: x.upper()),
    "length": RunnableLambda(len),
})

parallel.invoke('LangChain')

## 10. Structured output with Pydantic

Applications often need structured Python objects rather than free-form prose.

A parser can describe the desired schema and validate the model's response.

> Small local models are not always perfect JSON generators, so this section includes error handling.

In [ ]:
from pydantic import BaseModel, Field
from langchain_core.output_parsers import PydanticOutputParser


class ConceptCard(BaseModel):
    term: str = Field(description='The concept name')
    definition: str = Field(description='A concise definition')
    example: str = Field(description='A short example')


concept_parser = PydanticOutputParser(
    pydantic_object=ConceptCard
)

structured_prompt = PromptTemplate(
    template="""Return information about {topic}.
Follow these formatting instructions exactly:
{format_instructions}"""
    input_variables=['topic'],
    partial_variables={
        "format_instructions": concept_parser.get_format_instructions()
    },
)

structured_chain = (
    structured_prompt
    | hf_llm
    | concept_parser
)

In [ ]:
try:
    card = structured_chain.invoke({
        "topic": "cosine similarity"
    })
    print(card)
except Exception as exc:
    print('The local model did not produce valid structured output.')
    print('Try again or use a stronger model.')
    print(type(exc).__name__)

## 11. Documents and metadata

LangChain's `Document` object combines text with metadata.

Metadata is critical for RAG because it lets us preserve information such as source IDs, categories, dates, and URLs.

In [ ]:
from langchain_core.documents import Document

doc = Document(
    page_content="Standard returns are accepted within 14 days of delivery.",
    metadata={
        "source_id": "KB001",
        "category": "returns",
    },
)

print(doc.page_content)
print(doc.metadata)

## 12. Document loaders

LangChain provides loaders for many sources. Here we use a CSV loader.

The dataset is stored in the IAN 620 GitHub repository.

In [ ]:
DATA_URL = "https://raw.githubusercontent.com/ali2449/IAN-620/main/IAN_620_Module_9_Agentic_RAG_Knowledge_Base.csv"
CSV_FILE = "langchain_tutorial_kb.csv"

urllib.request.urlretrieve(DATA_URL, CSV_FILE)
print('Downloaded:', CSV_FILE)

In [ ]:
from langchain_community.document_loaders import CSVLoader

loader = CSVLoader(file_path=CSV_FILE)
loaded_rows = loader.load()

print('Loaded rows:', len(loaded_rows))
print(loaded_rows[0].page_content[:500])

### Manual document construction gives us more control

For many projects, building `Document` objects ourselves makes metadata handling clearer.

In [ ]:
kb = pd.read_csv(CSV_FILE)

documents = []

for _, row in kb.iterrows():
    documents.append(
        Document(
            page_content=(
                f"{row['title']}\n{row['text']}"
            ),
            metadata={
                'source_id': row['source_id'],
                'category': row['category'],
                'title': row['title'],
                'effective_date': row['effective_date'],
                'source_url': row['source_url'],
            },
        )
    )

print('Documents:', len(documents))
print(documents[0])

## 13. Text splitting

Long documents are usually split into smaller chunks before embedding.

`RecursiveCharacterTextSplitter` tries larger separators first, then progressively smaller ones.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=350,
    chunk_overlap=50,
)

chunks = splitter.split_documents(documents)

print('Original documents:', len(documents))
print('Chunks:', len(chunks))
print(chunks[0])

### Choosing chunk size

There is no universal best chunk size.

Smaller chunks can improve retrieval precision but may lose context. Larger chunks preserve context but can retrieve more irrelevant text.

Evaluate chunking with the types of questions your application actually receives.

## 14. Local embeddings

Embeddings map text into vectors so semantically related text can be compared numerically.

We use a local Sentence Transformers model.

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL,
    encode_kwargs={
        "normalize_embeddings": True
    },
)

vector = embeddings.embed_query(
    "return deadline after delivery"
)

print('Dimensions:', len(vector))
print('First 5 values:', vector[:5])

## 15. Chroma vector store

Chroma stores embedding vectors plus metadata.

LangChain's Chroma integration lives in the separate `langchain-chroma` package.

In [ ]:
from langchain_chroma import Chroma

CHROMA_DIR = "./langchain_tutorial_chroma"
COLLECTION = "langchain_tutorial"

shutil.rmtree(CHROMA_DIR, ignore_errors=True)

vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    ids=[
        f'doc-{i:03d}'
        for i in range(len(chunks))
    ],
    collection_name=COLLECTION,
    persist_directory=CHROMA_DIR,
)

print('Stored chunks:', len(vector_store.get()['ids']))

## 16. Similarity search

Start by testing the vector store directly before adding an LLM.

In [ ]:
query = "How many days do I have to return a normal item?"

results = vector_store.similarity_search(
    query,
    k=3,
)

for i, d in enumerate(results, start=1):
    print(f'--- Result {i} ---')
    print(d.metadata['source_id'])
    print(d.page_content)
    print()

### Similarity search with scores

Scores help with debugging, but their exact interpretation depends on the vector-store implementation and distance metric.

In [ ]:
scored = vector_store.similarity_search_with_score(
    query,
    k=3,
)

for doc, score in scored:
    print(
        doc.metadata['source_id'],
        'score =',
        round(float(score), 4),
    )

### Metadata filtering

Metadata can restrict retrieval to a known subset of the knowledge base.

In [ ]:
filtered = vector_store.similarity_search(
    "How long does a refund take?",
    k=3,
    filter={'category': 'refunds'},
)

[(d.metadata['source_id'], d.metadata['category']) for d in filtered]

## 17. Turn a vector store into a retriever

A retriever exposes a standard interface for returning relevant `Document` objects.

This makes the rest of the application less dependent on the specific vector database.

In [ ]:
retriever = vector_store.as_retriever(
    search_kwargs={'k': 3}
)

retrieved = retriever.invoke(
    "What happens if tracking has not changed?"
)

[(d.metadata['source_id'], d.metadata['title']) for d in retrieved]

## 18. Build classic RAG

Classic RAG is normally:

**question → retrieve → format context → prompt model → answer**

In [ ]:
def format_docs(docs):
    return '\n\n'.join(
        f"[{d.metadata.get('source_id', 'source')}] {d.page_content}"
        for d in docs
    )


rag_prompt = ChatPromptTemplate.from_template(
    """Answer the question using only the context below.
If the context is insufficient, say that you cannot verify the answer.
Cite source IDs such as [KB001].

CONTEXT:
{context}

QUESTION:
{question}"""
)

In [ ]:
def rag_answer(question):
    docs = retriever.invoke(question)
    context = format_docs(docs)

    chain = (
        rag_prompt
        | chat_model
        | StrOutputParser()
    )

    answer = chain.invoke({
        "context": context,
        "question": question,
    })

    return answer, docs

In [ ]:
answer, evidence = rag_answer(
    "Can I return a standard item after 20 days?"
)

print(answer)
print('\nSources:')
print([d.metadata['source_id'] for d in evidence])

## 19. Build the same RAG workflow with LCEL

We can make retrieval itself part of the Runnable pipeline.

In [ ]:
from langchain_core.runnables import RunnablePassthrough, RunnableLambda

rag_chain = (
    {
        "context": retriever | RunnableLambda(format_docs),
        "question": RunnablePassthrough(),
    }
    | rag_prompt
    | chat_model
    | StrOutputParser()
)

print(rag_chain.invoke(
    "How quickly should visible shipping damage be reported?"
))

### Why this is useful

The chain is now made from reusable Runnables. The retriever, prompt, model, and parser can each be replaced independently.

## 20. Tools with `@tool`

Tools are callable functions with well-defined inputs and outputs.

The model can use a tool's name, type hints, and docstring to understand what the tool does.

In [ ]:
from langchain.tools import tool


@tool
def add_numbers(a: float, b: float) -> float:
    """Add two numbers and return the result."""
    return a + b


print(add_numbers.name)
print(add_numbers.description)
print(add_numbers.args)

In [ ]:
add_numbers.invoke({
    "a": 12.5,
    "b": 7.5,
})

## 21. Turn the retriever into a tool

Once retrieval is a tool, an agent can decide whether it needs the knowledge base.

In [ ]:
@tool
def search_support_knowledge(query: str) -> str:
    """Search the approved internal support knowledge base.
Use this for returns, refunds, shipping, warranties, accounts,
billing, subscriptions, privacy, support hours, and related policies."""

    docs = retriever.invoke(query)
    return format_docs(docs)


print(search_support_knowledge.invoke({
    "query": "standard return window"
}))

## 22. DuckDuckGo web-search tool

DuckDuckGo search is useful for public/current information and does not require a separate search API key.

Do not use public web search as a replacement for approved internal policy.

In [ ]:
from langchain_community.tools import DuckDuckGoSearchRun

web_search = DuckDuckGoSearchRun()

try:
    result = web_search.invoke(
        "LangChain Python official documentation"
    )
    print(str(result)[:1500])
except Exception as exc:
    print('Web search is unavailable in this runtime.')
    print(type(exc).__name__)

## 23. Bind tools to a chat model

`bind_tools()` gives a tool-capable chat model the schemas of approved tools.

The model may then return `tool_calls` instead of a final answer.

> Small local models may not choose tools as reliably as larger models. The code is still useful for understanding the interface.

In [ ]:
tools = [
    add_numbers,
    search_support_knowledge,
    web_search,
]

try:
    model_with_tools = chat_model.bind_tools(tools)

    tool_request = model_with_tools.invoke(
        "Use the internal knowledge base to find the standard return window."
    )

    print('Content:', tool_request.content)
    print('Tool calls:', tool_request.tool_calls)
except Exception as exc:
    model_with_tools = None
    print('This local model/runtime did not complete tool binding.')
    print(type(exc).__name__)

## 24. What an agent loop actually does

A tool-using agent repeats this pattern:

1. Send the current messages to the model.
2. If the model requests a tool, execute the tool.
3. Add the tool result as a `ToolMessage`.
4. Send the updated messages back to the model.
5. Stop when the model returns a final answer or another stop condition is met.

`create_agent` automates this harness, but it is important to understand the underlying loop.

### Manual tool execution example

This example shows the mechanics directly.

In [ ]:
from langchain.messages import ToolMessage

if model_with_tools is not None:
    question = 'Use the calculator to add 18.5 and 7.25.'
    conversation = [HumanMessage(content=question)]

    ai_msg = model_with_tools.invoke(conversation)
    conversation.append(ai_msg)

    tool_map = {
        tool.name: tool
        for tool in tools
    }

    for call in ai_msg.tool_calls:
        selected_tool = tool_map[call['name']]
        tool_result = selected_tool.invoke(call['args'])

        conversation.append(
            ToolMessage(
                content=str(tool_result),
                tool_call_id=call['id'],
                name=call['name'],
            )
        )

    if ai_msg.tool_calls:
        final_msg = model_with_tools.invoke(conversation)
        print(final_msg.content)
    else:
        print(ai_msg.content)
else:
    print('Tool-calling model not available; continue to the agent section.')

## 25. LangChain `create_agent`

`create_agent` is the current high-level LangChain agent harness.

It combines:

- a chat model,
- tools,
- a system prompt,
- the model/tool execution loop,
- optional middleware and structured output.

Because this tutorial uses a very small local model, tool selection may be imperfect. A larger local tool-calling model or provider model will usually perform better.

In [ ]:
from langchain.agents import create_agent

agent = None

try:
    agent = create_agent(
        model=chat_model,
        tools=tools,
        system_prompt=(
            'You are a careful assistant. '
            'Use search_support_knowledge for internal support policy. '
            'Use web_search only for explicitly public/current information. '
            'Use add_numbers for arithmetic when useful.'
        ),
    )

    print('Agent created.')
except Exception as exc:
    print('The small local model could not initialize this agent.')
    print(type(exc).__name__)

In [ ]:
if agent is not None:
    try:
        result = agent.invoke({
            'messages': [
                {
                    "role": "user",
                    "content": "What is the standard return window in our internal policy?",
                }
            ]
        })

        print(result['messages'][-1].content)
    except Exception as exc:
        print('The local agent did not complete this tool-using run.')
        print('Try a larger local model or the Ollama example later.')
        print(type(exc).__name__)

## 26. Structured output from an agent

Current LangChain agents can return validated structured responses when the selected model supports the required structured-output strategy.

A Pydantic model is a convenient schema.

In [ ]:
class SupportDecision(BaseModel):
    category: str
    answer: str
    needs_human: bool


try:
    structured_agent = create_agent(
        model=chat_model,
        tools=[search_support_knowledge],
        response_format=SupportDecision,
        system_prompt=(
            'Use the internal knowledge tool when necessary. '
            'Return a structured support decision.'
        ),
    )

    structured_result = structured_agent.invoke({
        'messages': [
            {
                "role": "user",
                "content": "Can a standard item be returned 20 days after delivery?",
            }
        ]
    })

    print(structured_result.get('structured_response'))
except Exception as exc:
    print('Structured agent output needs a model with reliable structured/tool support.')
    print(type(exc).__name__)

## 27. Conversation history

LangChain can wrap a Runnable so previous messages are stored and reused for a session.

This is useful for conversational applications. It is different from long-term memory.

In [ ]:
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.prompts import MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory

conversation_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are a concise assistant. Use conversation history when useful.",
    ),
    MessagesPlaceholder(variable_name='history'),
    (
        "human",
        "{question}",
    ),
])

conversation_chain = (
    conversation_prompt
    | chat_model
    | StrOutputParser()
)

history_store = {}


def get_history(session_id):
    if session_id not in history_store:
        history_store[session_id] = InMemoryChatMessageHistory()
    return history_store[session_id]


chat_with_history = RunnableWithMessageHistory(
    conversation_chain,
    get_history,
    input_messages_key='question',
    history_messages_key='history',
)

In [ ]:
config = {"configurable": {"session_id": "demo-user"}}

print(chat_with_history.invoke(
    {"question": "My favorite vector database in this tutorial is Chroma."},
    config=config,
))

print(chat_with_history.invoke(
    {"question": "Which vector database did I say I liked?"},
    config=config,
))

## 28. Optional: local Ollama + LangChain

For a laptop or desktop, Ollama is often the easiest way to run a larger local chat model.

Unlike the core Colab path, this section assumes Ollama is installed and running on your own computer.

Example terminal setup:

```bash
ollama pull qwen3:4b
pip install -U langchain-ollama
```

Then LangChain can use it through `ChatOllama`.

In [ ]:
# Run this only on a machine where Ollama is installed and running.

OLLAMA_EXAMPLE = r'''
from langchain_ollama import ChatOllama

ollama_model = ChatOllama(
    model="qwen3:4b",
    temperature=0,
)

print(ollama_model.invoke(
    "Explain LangChain in one paragraph."
).content)
'''

print(OLLAMA_EXAMPLE)

### Ollama tool calling

Many Ollama chat models support tool calling. The LangChain pattern is the same:

```python
ollama_with_tools = ollama_model.bind_tools(tools)
response = ollama_with_tools.invoke('Use the internal policy search tool...')
print(response.tool_calls)
```

A larger local model generally makes better tool-routing decisions than a very small teaching model.

## 29. Optional provider models

LangChain supports many providers. Provider models can be useful when you need stronger reasoning, robust structured output, or high-quality tool calling.

With current LangChain, model identifiers can often be passed as `provider:model` strings to `create_agent` after the provider integration package is installed and configured.

Examples include OpenAI, Anthropic, Google, OpenRouter, Fireworks, Ollama, and others.

Keep provider-specific code at the edge of your application so the rest of the workflow remains portable.

## 30. LangSmith tracing — optional

LangSmith is the LangChain ecosystem's tracing/evaluation platform.

Tracing is optional for this tutorial, but it becomes very useful when an application contains:

- multiple prompts,
- retriever calls,
- tool calls,
- retries,
- agents,
- long execution traces.

Typical setup:

```python
os.environ['LANGSMITH_TRACING'] = 'true'
os.environ['LANGSMITH_API_KEY'] = '...'
```

Do not hard-code secrets in notebooks committed to GitHub.

## 31. Where LangGraph begins

LangChain's high-level agent harness is useful when the standard model/tool loop is enough.

Use LangGraph when you want explicit control such as:

- a state schema,
- deterministic nodes,
- model nodes,
- conditional branches,
- loops,
- maximum-step policies,
- persistence/checkpoints,
- human approval,
- resumable workflows.

Conceptually, a LangChain retriever, model, or tool can simply become a function used inside a LangGraph node.

In [ ]:
LANGGRAPH_SKETCH = r'''
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

class State(TypedDict):
    question: str
    documents: list
    answer: str

def retrieve_node(state):
    docs = retriever.invoke(state['question'])
    return {'documents': docs}

def generate_node(state):
    context = format_docs(state['documents'])
    answer = rag_chain.invoke(state['question'])
    return {'answer': answer}

builder = StateGraph(State)
builder.add_node('retrieve', retrieve_node)
builder.add_node('generate', generate_node)
builder.add_edge(START, 'retrieve')
builder.add_edge('retrieve', 'generate')
builder.add_edge('generate', END)

graph = builder.compile()
'''

print(LANGGRAPH_SKETCH)

## 32. Reload a persisted Chroma collection

Persistence lets an application reuse the vector database without re-embedding every document.

In [ ]:
reloaded_store = Chroma(
    collection_name=COLLECTION,
    persist_directory=CHROMA_DIR,
    embedding_function=embeddings,
)

print('Reloaded vectors:', len(reloaded_store.get()['ids']))

reloaded_store.similarity_search(
    "How long is the warranty?",
    k=2,
)

## 33. Small retrieval evaluation

Evaluate retrieval independently from generation.

If retrieval fails, changing the generation prompt will not fix the underlying evidence problem.

In [ ]:
tests = [
    ('What is the normal return deadline?', 'returns'),
    ('How long does refund processing take?', 'refunds'),
    ('When should shipping damage be reported?', 'damaged_items'),
    ('How long is the limited warranty?', 'warranty'),
    ('How can I reset my password?', 'accounts'),
]

rows = []

for question, expected in tests:
    docs = retriever.invoke(question)
    categories = [
        d.metadata.get('category', '')
        for d in docs
    ]

    rows.append({
        'question': question,
        'expected_category': expected,
        'retrieved_categories': ', '.join(categories),
        'hit': expected in categories,
    })

evaluation = pd.DataFrame(rows)
evaluation

In [ ]:
print(
    'Retrieval hit rate:',
    f"{evaluation['hit'].mean():.0%}"
)

## 34. Common mistakes

### Mistake 1: Starting with agents before understanding the components
Build and test the model, retriever, and tools separately first.

### Mistake 2: Treating LangChain as required
You can always write the same logic in normal Python. Use LangChain where its abstractions reduce integration work.

### Mistake 3: Mixing retrieval and generation errors
Inspect retrieved documents before debugging the model prompt.

### Mistake 4: Giving agents too many tools
Use the smallest tool set needed for the task.

### Mistake 5: Using public web search as internal truth
Keep authoritative internal sources and public web evidence separate.

### Mistake 6: Using a tiny local model for production-grade tool routing
Small models are excellent for learning, but larger or specialized models usually make better tool decisions.

### Mistake 7: Copying old tutorials without checking imports
LangChain evolves quickly. Prefer current `create_agent`, Runnable/LCEL patterns, and integration packages.

## 35. LangChain learning map

| Level | What to master |
|---|---|
| 1 | model `.invoke()` and messages |
| 2 | prompt templates and output parsers |
| 3 | LCEL / Runnables |
| 4 | `Document`, loaders, splitters |
| 5 | embeddings, vector stores, retrievers |
| 6 | classic RAG |
| 7 | `@tool` and direct tool execution |
| 8 | tool binding |
| 9 | `create_agent` |
| 10 | structured output, history, tracing |
| 11 | LangGraph for custom orchestration |

## 36. Final architecture

You have now worked through the main LangChain stack:

**Local model → messages → prompts → Runnables → parsers → Documents → splitters → embeddings → Chroma → retriever → RAG → tools → DuckDuckGo → tool calling → agent → history → structured output → LangGraph bridge**

The most important skill is not memorizing imports. It is knowing which abstraction solves which problem.

### Official references

- LangChain overview: https://docs.langchain.com/oss/python/langchain/overview
- Agents: https://docs.langchain.com/oss/python/langchain/agents
- Tools: https://docs.langchain.com/oss/python/langchain/tools
- Chroma integration: https://docs.langchain.com/oss/python/integrations/vectorstores/chroma
- Hugging Face local pipelines: https://docs.langchain.com/oss/python/integrations/llms/huggingface_pipelines
- ChatHuggingFace: https://docs.langchain.com/oss/python/integrations/chat/huggingface
- ChatOllama: https://docs.langchain.com/oss/python/integrations/chat/ollama
- LangGraph overview: https://docs.langchain.com/oss/python/langgraph/overview